# QueryStation Remote DuckDB Explorer

Query DuckLake and Iceberg catalogs via the QueryStation Arrow IPC API. Setup in uour terminal first. The python package ipykernel will let you use your vemv here.

**Setup:**
```bash
uv venv
Follow instructions and activate
uv sync --extra notebook
```

**Auth:** Set `QUERYSTATION_API_KEY` and `AUTH_URL` in your `.env` file at the repo root.

In [1]:
# ── Setup ────────────────────────────────────────────────
import polars as pl
from IPython.display import display, HTML
from dotenv import load_dotenv
from data_consumers import RemoteDuckDBWrapper

load_dotenv()

db = RemoteDuckDBWrapper()


# ── Notebook display helpers ──────────────────────────────
def show(df: pl.DataFrame, max_rows: int = 200):
    """Render a scrollable HTML table with sticky headers."""
    pdf = df.head(max_rows).to_pandas()
    table_html = pdf.to_html(index=False, border=0, classes=["dataframe"])
    display(HTML(f"""
    <div style="max-height:500px; overflow-y:auto; overflow-x:auto; border:1px solid #444;">
        <style>
            .dataframe thead th {{ position:sticky; top:0; background:#222; color:white; z-index:1; }}
            .dataframe td {{ white-space:nowrap; font-size:13px; }}
        </style>
        {table_html}
    </div>
    <p style="color:#888; font-size:12px;">{df.height} rows x {df.width} cols{f' (showing {max_rows})' if df.height > max_rows else ''}</p>
    """))


# ── Convenience wrappers ──────────────────────────────────
def sql(query: str) -> pl.DataFrame:
    """Shorthand for db.sql()."""
    return db.sql(query)


def catalog():
    """Show all catalogs, schemas, and tables."""
    show(db.catalog())


def describe(table: str):
    """Show column names and types for a table."""
    df = db.describe(table)
    if df.is_empty():
        print(f"Table not found: {table}")
        return
    print(f"Schema: {table} ({df.height} columns)")
    show(df)


def export(df: pl.DataFrame, name: str, fmt: str = "csv", output_dir: str = "../data/exports"):
    """Export a DataFrame to parquet, csv, or json."""
    path = RemoteDuckDBWrapper.export(df, name, fmt, output_dir)
    print(f"Exported {df.height} rows to {path}")


print("Ready. Use sql(), show(), catalog(), describe(), export().")

Ready. Use sql(), show(), catalog(), describe(), export().


## Connect & Browse Catalog

In [2]:
catalog()

catalog,schema,name,columns
iceberg,default,people,1
iceberg,nyc_operations,service_requests_311,1
lake,nyc_checkbook,checkbook_budget,21
lake,nyc_checkbook,checkbook_contracts_reg_expense,54
lake,nyc_checkbook,checkbook_revenue,25
lake,nyc_checkbook,checkbook_revenue_nycha,23
lake,nyc_checkbook,checkbook_spending,40
lake,nyc_environment,floodnet_events_joined,43
lake,nyc_environment,floodnet_flooding_events,13
lake,nyc_environment,floodnet_sensor_metadata,16


## Describe a Table

In [3]:
describe("lake.nyc_operations.service_requests_311")

Schema: lake.nyc_operations.service_requests_311 (58 columns)


column_name,column_type
unique_key,VARCHAR
created_date,TIMESTAMP WITH TIME ZONE
closed_date,TIMESTAMP WITH TIME ZONE
agency_acronym,VARCHAR
agency_name,VARCHAR
complaint_type,VARCHAR
descriptor,VARCHAR
descriptor_2,VARCHAR
location_type,VARCHAR
incident_zip,VARCHAR


In [ ]:
describe("lake.nyc_finance.capital_budget")

## Query Examples

### 311 Service Requests

In [4]:
# Top complaint types in 2025
df = sql("""
SELECT complaint_type, COUNT(*) AS cnt
FROM lake.nyc_operations.service_requests_311
WHERE year = 2025
GROUP BY 1
ORDER BY 2 DESC
LIMIT 15
""")
show(df)

complaint_type,cnt
Illegal Parking,577257
Noise - Residential,463349
HEAT/HOT WATER,315946
Noise - Street/Sidewalk,173049
Blocked Driveway,172723
UNSANITARY CONDITION,117720
Water System,77517
Street Condition,70330
PLUMBING,69535
Abandoned Vehicle,67903


In [ ]:
# Complaints by borough
df = sql("""
SELECT borough, COUNT(*) AS total,
       SUM(CASE WHEN is_overnight THEN 1 ELSE 0 END) AS overnight,
       ROUND(100.0 * SUM(CASE WHEN is_overnight THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_overnight
FROM lake.nyc_operations.service_requests_311
WHERE year = 2025
GROUP BY 1
ORDER BY 2 DESC
""")
show(df)

In [ ]:
# Full row sample (scrollable)
df = sql("SELECT * FROM lake.nyc_operations.service_requests_311 LIMIT 100")
show(df)

### Capital Budget

In [ ]:
# Awards by funding category
df = sql("""
SELECT funding_category, COUNT(*) AS projects, ROUND(SUM(award), 2) AS total_award
FROM lake.nyc_finance.capital_budget
GROUP BY 1
ORDER BY 3 DESC
LIMIT 10
""")
show(df)

### City Payroll

In [ ]:
# Top agencies by total pay
df = sql("""
SELECT agency_name, COUNT(*) AS employees, ROUND(SUM(regular_gross_paid), 2) AS total_pay
FROM lake.nyc_finance.city_payroll
GROUP BY 1
ORDER BY 3 DESC
LIMIT 10
""")
show(df)

### FloodNet

In [ ]:
# Sensors by borough
df = sql("""
SELECT borough, COUNT(*) AS sensors
FROM lake.nyc_environment.floodnet_sensor_metadata
GROUP BY 1
ORDER BY 2 DESC
""")
show(df)

### MTA Ridership

In [ ]:
df = sql("SELECT * FROM lake.nys_transportation.mta_daily_ridership LIMIT 20")
show(df)

### Iceberg

In [ ]:
df = sql("SELECT * FROM iceberg.default.people")
show(df)

---
## Your Queries

Write SQL, get a DataFrame, render it scrollable.

In [ ]:
df = sql("""

SELECT 42 AS answer

""")
show(df)

## Export Results

Export any DataFrame to `data/exports/` as CSV, Parquet, or JSON.

In [ ]:
# Example: export the last query result
# export(df, "my_export", "csv")
# export(df, "my_export", "parquet")
# export(df, "my_export", "json")